# 08 - Lesion Detection Model (YOLOv8)

Train a YOLOv8-small lesion detector for identifying skin lesion types.

## Datasets
- **ACNE04** - acne lesion detection with bounding boxes
- **ISIC 2018** - dermoscopic lesion images with segmentation masks
- **HAM10000** - 10,015 dermatoscopic images with diagnosis labels
- **DDI** (Diverse Dermatology Images) - diverse skin tone representation

## Classes
- comedone, papule, pustule, nodule, macule, patch

## Training Strategy
1. Pretrain on COCO (use YOLOv8s pretrained weights)
2. Fine-tune on HAM10000 + ISIC for general lesion detection
3. Fine-tune on ACNE04 for acne-specific lesion types

## Output
- ONNX export for backend inference
- CoreML export for on-device inference (iOS)

In [ ]:
# Install dependencies (Colab)
!pip install -q ultralytics onnx onnxruntime coremltools opencv-python albumentations

In [ ]:
import os
import json
import shutil
import yaml
import numpy as np
import cv2
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Lesion classes
CLASSES = ["comedone", "papule", "pustule", "nodule", "macule", "patch"]
NUM_CLASSES = len(CLASSES)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {CLASSES}")

## Dataset Preparation

Convert all datasets to YOLO format (one txt file per image with class_id, x_center, y_center, width, height).
We merge HAM10000, ISIC, ACNE04, and DDI into a unified dataset.

In [ ]:
DATA_ROOT = Path("./data/lesion_detection")


def create_dataset_yaml(data_root: Path, classes: list) -> str:
    """Create YOLO dataset configuration YAML."""
    config = {
        "path": str(data_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(classes),
        "names": classes,
    }
    yaml_path = data_root / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    print(f"Created dataset config: {yaml_path}")
    return str(yaml_path)


def convert_bbox_to_yolo(img_w: int, img_h: int, x1: int, y1: int, x2: int, y2: int) -> tuple:
    """Convert (x1, y1, x2, y2) bounding box to YOLO format."""
    x_center = ((x1 + x2) / 2) / img_w
    y_center = ((y1 + y2) / 2) / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h
    return x_center, y_center, width, height


def prepare_ham10000_isic(source_dir: Path, output_dir: Path, class_mapping: dict):
    """Convert HAM10000/ISIC annotations to YOLO format.

    class_mapping: dict mapping diagnosis string to class_id.
    """
    images_dir = output_dir / "images" / "train"
    labels_dir = output_dir / "labels" / "train"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    metadata_path = source_dir / "HAM10000_metadata.csv"
    if not metadata_path.exists():
        print(f"Metadata not found at {metadata_path}, skipping HAM10000 prep.")
        return

    import pandas as pd
    df = pd.read_csv(metadata_path)
    count = 0
    for _, row in df.iterrows():
        img_id = row["image_id"]
        dx = row["dx"]
        if dx not in class_mapping:
            continue

        src_img = source_dir / "images" / f"{img_id}.jpg"
        if not src_img.exists():
            continue

        shutil.copy(src_img, images_dir / f"{img_id}.jpg")

        # HAM10000 has full-image labels; use full image as bounding box
        class_id = class_mapping[dx]
        label_path = labels_dir / f"{img_id}.txt"
        with open(label_path, "w") as f:
            f.write(f"{class_id} 0.5 0.5 0.9 0.9\n")
        count += 1

    print(f"Prepared {count} HAM10000/ISIC samples.")


def prepare_acne04(source_dir: Path, output_dir: Path):
    """Convert ACNE04 annotations to YOLO format."""
    images_dir = output_dir / "images" / "train"
    labels_dir = output_dir / "labels" / "train"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    ann_dir = source_dir / "annotations"
    if not ann_dir.exists():
        print(f"ACNE04 annotations not found at {ann_dir}, skipping.")
        return

    count = 0
    for ann_file in ann_dir.glob("*.xml"):
        import xml.etree.ElementTree as ET
        tree = ET.parse(ann_file)
        root = tree.getroot()
        filename = root.find("filename").text
        size = root.find("size")
        img_w = int(size.find("width").text)
        img_h = int(size.find("height").text)

        src_img = source_dir / "images" / filename
        if src_img.exists():
            shutil.copy(src_img, images_dir / filename)

        label_lines = []
        for obj in root.findall("object"):
            name = obj.find("name").text.lower()
            if name in [c.lower() for c in CLASSES]:
                class_id = [c.lower() for c in CLASSES].index(name)
                bbox = obj.find("bndbox")
                x1 = int(bbox.find("xmin").text)
                y1 = int(bbox.find("ymin").text)
                x2 = int(bbox.find("xmax").text)
                y2 = int(bbox.find("ymax").text)
                xc, yc, w, h = convert_bbox_to_yolo(img_w, img_h, x1, y1, x2, y2)
                label_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        if label_lines:
            label_path = labels_dir / f"{Path(filename).stem}.txt"
            with open(label_path, "w") as f:
                f.write("\n".join(label_lines) + "\n")
            count += 1

    print(f"Prepared {count} ACNE04 samples.")


# Uncomment to run data preparation
# HAM10000_MAPPING = {"bcc": 4, "mel": 4, "bkl": 4, "nv": 4, "akiec": 3, "vasc": 4, "df": 5}
# prepare_ham10000_isic(Path("./data/ham10000"), DATA_ROOT, HAM10000_MAPPING)
# prepare_acne04(Path("./data/acne04"), DATA_ROOT)
# dataset_yaml = create_dataset_yaml(DATA_ROOT, CLASSES)

## Phase 1: Fine-tune on HAM10000 + ISIC

Start from COCO-pretrained YOLOv8s weights and fine-tune on general dermatological
lesion images for broad lesion detection capability.

In [ ]:
# Phase 1: General lesion detection fine-tuning
# model_phase1 = YOLO("yolov8s.pt")  # COCO-pretrained YOLOv8-small

# results_phase1 = model_phase1.train(
#     data=str(DATA_ROOT / "dataset.yaml"),
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     lr0=1e-3,
#     lrf=0.01,
#     warmup_epochs=5,
#     augment=True,
#     hsv_h=0.015,
#     hsv_s=0.4,
#     hsv_v=0.3,
#     flipud=0.5,
#     fliplr=0.5,
#     mosaic=1.0,
#     mixup=0.1,
#     project="runs/lesion",
#     name="phase1_general",
#     patience=10,
# )

print("Phase 1 training configured (uncomment to run).")

## Phase 2: Fine-tune on ACNE04

Continue training from Phase 1 weights on the ACNE04 dataset to specialize
in acne lesion detection (comedone, papule, pustule, nodule).

In [ ]:
# Phase 2: Acne-specific fine-tuning
# best_phase1 = "runs/lesion/phase1_general/weights/best.pt"
# model_phase2 = YOLO(best_phase1)

# results_phase2 = model_phase2.train(
#     data=str(DATA_ROOT / "dataset.yaml"),
#     epochs=30,
#     imgsz=640,
#     batch=16,
#     lr0=5e-4,
#     lrf=0.01,
#     warmup_epochs=3,
#     freeze=10,  # Freeze first 10 layers for fine-tuning
#     augment=True,
#     project="runs/lesion",
#     name="phase2_acne",
#     patience=10,
# )

print("Phase 2 training configured (uncomment to run).")

## Evaluation

In [ ]:
def evaluate_model(model_path: str, data_yaml: str):
    """Run validation and print per-class metrics."""
    model = YOLO(model_path)
    results = model.val(data=data_yaml, imgsz=640, batch=16)

    print(f"\nmAP@0.5: {results.box.map50:.4f}")
    print(f"mAP@0.5:0.95: {results.box.map:.4f}")
    print(f"\nPer-class AP@0.5:")
    for i, cls_name in enumerate(CLASSES):
        if i < len(results.box.ap50):
            print(f"  {cls_name}: {results.box.ap50[i]:.4f}")
    return results


# Uncomment to evaluate
# results = evaluate_model("runs/lesion/phase2_acne/weights/best.pt", str(DATA_ROOT / "dataset.yaml"))

## Export: ONNX + CoreML

Export the final model to both ONNX (backend inference) and CoreML (iOS on-device).

In [ ]:
def export_model(model_path: str):
    """Export trained model to ONNX and CoreML formats."""
    model = YOLO(model_path)

    # ONNX export
    onnx_path = model.export(format="onnx", imgsz=640, simplify=True, dynamic=True)
    print(f"ONNX exported to: {onnx_path}")

    # Verify ONNX
    import onnxruntime as ort
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name
    dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
    output = session.run(None, {input_name: dummy})
    print(f"ONNX inference OK, output shape: {output[0].shape}")

    # CoreML export
    coreml_path = model.export(format="coreml", imgsz=640)
    print(f"CoreML exported to: {coreml_path}")

    return onnx_path, coreml_path


# Uncomment to export
# onnx_path, coreml_path = export_model("runs/lesion/phase2_acne/weights/best.pt")

## Inference Demo

In [ ]:
def run_inference_demo(model_path: str, image_path: str):
    """Run inference on a sample image and visualize detections."""
    model = YOLO(model_path)
    results = model(image_path, imgsz=640, conf=0.25)

    for r in results:
        boxes = r.boxes
        print(f"Detected {len(boxes)} lesions:")
        for box in boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            xyxy = box.xyxy[0].tolist()
            print(f"  {CLASSES[cls_id]}: conf={conf:.3f}, bbox={[f'{v:.0f}' for v in xyxy]}")

        # Visualize
        annotated = r.plot()
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title("Lesion Detections")
        plt.show()


# Uncomment to run demo
# run_inference_demo("runs/lesion/phase2_acne/weights/best.pt", "./data/test_face.jpg")